In [33]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem


In [34]:
DATA_PATH = "dataset.xls"   # change name if needed

data = pd.read_excel(DATA_PATH)
print("Full dataset shape:", data.shape)
print("Columns:", data.columns.tolist())


Full dataset shape: (17571, 7)
Columns: ['SMILES', 'CO2', 'O2', 'N2', 'CO2/N2', 'CO2/O2', 'Type']


In [35]:
smiles_col = "SMILES"
target_cols = ["CO2", "N2", "O2"]

data = data[[smiles_col] + target_cols]
print("Selected data shape:", data.shape)


Selected data shape: (17571, 4)


In [36]:
data["log_CO2"] = np.log10(data["CO2"])
data["log_N2"]  = np.log10(data["N2"])
data["log_O2"]  = np.log10(data["O2"])


In [37]:
before = data.shape[0]

data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=["log_CO2", "log_N2", "log_O2", smiles_col])

after = data.shape[0]

print(f"Removed {before - after} invalid rows")
print("Cleaned data shape:", data.shape)


Removed 1 invalid rows
Cleaned data shape: (17570, 7)


In [38]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol

data["mol"] = data[smiles_col].apply(smiles_to_mol)

# Remove invalid molecules
data = data[data["mol"].notnull()].reset_index(drop=True)

print("Valid molecules:", data.shape[0])


Valid molecules: 17570


In [39]:
def smiles_to_ecfp(mol, radius=2, n_bits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=radius,
        nBits=n_bits
    )
    arr = np.zeros((n_bits,), dtype=int)
    AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


In [40]:
X_ecfp_full = np.vstack(
    data["mol"].apply(smiles_to_ecfp)
)

print("ECFP matrix shape:", X_ecfp_full.shape)


ECFP matrix shape: (17570, 2048)


In [41]:
y_full = data[["log_CO2", "log_N2", "log_O2"]].reset_index(drop=True)

print("Target shape:", y_full.shape)


Target shape: (17570, 3)


In [42]:
assert X_ecfp_full.shape[0] == y_full.shape[0], "Mismatch between X and y"

print("NaNs in X:", np.isnan(X_ecfp_full).sum())
print("NaNs in y:\n", y_full.isna().sum())


NaNs in X: 0
NaNs in y:
 log_CO2    0
log_N2     0
log_O2     0
dtype: int64


In [43]:
np.save("X_ecfp_full.npy", X_ecfp_full)
y_full.to_csv("y_targets_full.csv", index=False)

print("Saved:")
print(" - X_ecfp_full.npy")
print(" - y_targets_full.csv")


Saved:
 - X_ecfp_full.npy
 - y_targets_full.csv


In [44]:
print("Notebook C summary")
print("------------------")
print("Dataset: Experimental + GA")
print("Samples:", X_ecfp_full.shape[0])
print("Fingerprint size:", X_ecfp_full.shape[1])
print("Targets:", list(y_full.columns))


Notebook C summary
------------------
Dataset: Experimental + GA
Samples: 17570
Fingerprint size: 2048
Targets: ['log_CO2', 'log_N2', 'log_O2']
